# Playwright Basics for Agents [Agent Patterns - Module 10]

> **MLCourse - Agentic AI - Agent Patterns**

A **browser agent** is an agent whose tools drive a real web browser: open a
page, read it, click things, type into fields. That is genuinely useful -
most of the world's systems have a web UI and no API - and it is also the
most dangerous tool shape in this whole course, because the page content is
written by someone else.

This module teaches the mechanism at its smallest. **Everything runs
against local HTML files on disk.** No network, no live sites, no
non-determinism, and nothing that could ever accidentally act on a real
service.

### What you will learn

1. What Playwright is and the three objects that matter.
2. Creating local HTML fixtures and opening them with `file://`.
3. The two ways an agent can "see" a page: raw HTML vs rendered text.
4. Why you almost always feed the model rendered text, not HTML.
5. Locators - how you address an element without guessing.

### Key takeaways

- Playwright is a normal Python library driving a real Chromium process.
- `page.inner_text("body")` is the agent's eyes; keep it small.
- Local fixtures make browser work testable. Use them in your own projects.

### Prerequisite

Chromium must be downloaded once:

```bash
python -m playwright install chromium
```

It is already installed in this environment.

### Setup: imports, environment, track discovery


In [ ]:
import os
import re
import sys
import json
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

HERE = Path.cwd().resolve()                  # the module folder
FIXTURES = HERE / "fixtures"
FIXTURES.mkdir(exist_ok=True)

print(f"Track root : {TRACK}")
print(f"Module dir : {HERE}")
print(f"Model      : {MODEL} (via Groq)")


### 1. The three Playwright objects

```
sync_playwright()  ->  the driver process
   .chromium.launch()  ->  Browser      (a real Chromium)
      .new_page()      ->  Page         (a tab; this is what you automate)
```

Always close the browser. In a notebook, a leaked Chromium process survives
the cell and eats memory until the kernel dies.

There is one wrinkle that only bites inside Jupyter, and the next cell deals
with it once for the whole module - read the comments there, because the
error message it prevents ("using Playwright Sync API inside the asyncio
loop") is one you will meet again.

### Running Playwright inside Jupyter (Windows)


In [ ]:
# Two environment quirks, handled once here and reused in every notebook:
#
# 1. Jupyter's kernel already runs an asyncio event loop in the main thread.
#    Playwright's SYNC api refuses to start inside a running loop, so we run
#    every browser job in a short-lived worker thread.
# 2. ipykernel installs the Selector event-loop policy on Windows, and that
#    policy cannot spawn subprocesses - which is exactly what launching
#    Chromium needs. We restore the Proactor policy so new loops can.
#
# In a plain .py script neither applies: `with sync_playwright() as p:` just works.

import asyncio
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

def browse(url, job, headless=True):
    """Open `url` in Chromium, hand the Page to job(page), return its result."""
    def _run():
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=headless)
            page = browser.new_page()
            page.goto(url)
            try:
                return job(page)
            finally:
                browser.close()          # never leak a Chromium process
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(_run).result()

print("browse() ready - policy:", type(asyncio.get_event_loop_policy()).__name__)


### Write the local fixture page


In [ ]:
# Everything in this module runs against files on disk. No network, ever.
# That makes the notebook deterministic AND safe to re-run offline.

CATALOG = FIXTURES / "catalog.html"
CATALOG.write_text("""<!doctype html>
<html><head><meta charset="utf-8"><title>Northwind Parts - Catalog</title></head>
<body>
  <h1>Northwind Parts</h1>
  <p class="tagline">Industrial fittings, shipped same day.</p>

  <table id="products">
    <tr><th>SKU</th><th>Name</th><th>Price</th><th>Stock</th></tr>
    <tr><td>NW-1001</td><td>Brass elbow 15mm</td><td>3.40</td><td>184</td></tr>
    <tr><td>NW-1002</td><td>Copper tee 22mm</td><td>5.95</td><td>0</td></tr>
    <tr><td>NW-1007</td><td>PTFE tape 12m</td><td>0.85</td><td>1620</td></tr>
    <tr><td>NW-1044</td><td>Stainless clamp 40mm</td><td>2.10</td><td>37</td></tr>
    <tr><td>NW-2100</td><td>Compression valve 15mm</td><td>11.25</td><td>6</td></tr>
  </table>

  <h2>Delivery</h2>
  <p>Orders placed before 14:00 ship the same working day.
     Free delivery over 50 GBP.</p>

  <footer><a href="contact.html" id="contact-link">Contact us</a></footer>
</body></html>
""", encoding="utf-8")

print("wrote", CATALOG)
print("size:", CATALOG.stat().st_size, "bytes")


### Open the local page and look at it


In [ ]:
url = CATALOG.as_uri()          # file:///D:/.../fixtures/catalog.html
print("opening:", url)

def peek(page):
    return {
        "title": page.title(),
        "h1": page.inner_text("h1"),
        "rows": page.locator("#products tr").count(),
    }

info = browse(url, peek)

print("title     :", info["title"])
print("h1        :", info["h1"])
print("table rows:", info["rows"], "(including the header row)")


`headless=True` (the default in our `browse` helper) means no window opens.
That is what you want on a server, in CI, and in a notebook. Pass
`headless=False` on your own machine when you are debugging and want to
*watch* the agent work - it is the single most useful debugging trick in
browser automation.

### 2. HTML vs rendered text

An agent needs the page as *text* to put in a prompt. You have two options:

- `page.content()` - the full HTML source. Complete, but enormous, full of
  `<div class="css-1x7bfz">` noise, and mostly tokens you pay for and the
  model ignores.
- `page.inner_text("body")` - what a human would see. Compact, ordered,
  already stripped of markup.

For a real site the difference is often 50x. Look at it.

### Compare the two views


In [ ]:
def both_views(page):
    return page.content(), page.inner_text("body")

html, text = browse(CATALOG.as_uri(), both_views)

print(f"page.content()          : {len(html):>6} chars")
print(f"page.inner_text('body') : {len(text):>6} chars")
print(f"ratio                   : {len(html)/len(text):.1f}x")
print()
print("--- rendered text -------------------------------------------------")
print(text)


Even on this tiny hand-written fixture the HTML is several times larger. On
a real page with a framework, analytics and inline SVG, `page.content()` can
be 200 KB - far past a sane prompt budget.

**Rule:** send rendered text. Reach for HTML only when you specifically need
attributes the text does not carry (an `href`, a hidden `value`, a
`data-id`), and then extract just those.

### 3. Locators

A **locator** is Playwright's way of addressing an element. It is lazy - it
describes *how to find* the element, and resolves when you use it - which is
why Playwright can auto-wait for elements instead of you writing `sleep()`.

Preference order, best first:

1. **Role / label / text** - `get_by_role("button", name="Send")`. Survives
   redesigns, and it is how a human describes the element.
2. **id** - `#submit`. Stable when the app author gave it one.
3. **CSS / XPath structure** - `div > div:nth-child(3) > span`. Brittle.
   Any layout change breaks it.

An agent that emits selectors will emit bad ones. Prefer to give the model a
*menu* of pre-written locators (notebook 03) rather than letting it invent
CSS.

### Extract the product table with locators


In [ ]:
def read_table(page):
    rows = page.locator("#products tr")
    n = rows.count()
    header = [c.strip() for c in rows.nth(0).inner_text().split("\t")]
    products = []
    for i in range(1, n):
        cells = [c.strip() for c in rows.nth(i).inner_text().split("\t")]
        products.append(dict(zip(header, cells)))
    link = page.locator("#contact-link").get_attribute("href")
    return header, products, link

header, products, link = browse(CATALOG.as_uri(), read_table)

print("header:", header)
for prod in products:
    print(" ", prod)
print("\nfooter link href:", link, "  <-- only an attribute lookup gets this")


### Was the model needed for that?

No - and that is an important point to make early. The table has a fixed
structure, so a loop over locators is **cheaper, faster and 100% accurate**.
Do not put an LLM in the middle of a job a selector already does.

The model earns its place when the structure is *unknown or irregular*:
prose you must pull facts out of, layouts that differ per page, fields whose
names vary. That is notebook 02.

### Screenshots: the other debugging trick


In [ ]:
shot = HERE / "catalog.png"

def snap(page):
    page.screenshot(path=str(shot), full_page=True)
    return shot.stat().st_size

size = browse(CATALOG.as_uri(), snap)
print("saved", shot.name, "-", size, "bytes")


Screenshots are how you find out why a headless run behaved differently from
what you expected. They are also the input to *vision* browser agents, which
reason over the rendered pixels instead of the text - more capable, far more
expensive, and out of scope here.

### Pitfalls recap

- **Leaked browsers.** Always close in a `finally`, as `browse()` does.
- **`page.content()` in a prompt.** You will blow your context window on
  `<div>` noise. Use `inner_text`.
- **Structural CSS selectors.** `div:nth-child(3)` breaks on the next
  redesign. Prefer role/label/id.
- **The Jupyter asyncio clash.** Sync Playwright inside a notebook needs a
  worker thread (and, on Windows, the Proactor loop policy). In a plain
  script, none of that applies.
- **`sleep()` for timing.** Playwright auto-waits on locators. Extra sleeps
  hide races instead of fixing them.

### Next

Notebook 02 hands the rendered text to Groq and asks for structured data
back.